# NB36 — EX-3 Typed EventEmitter + K-3/K-7 Closure

**Objective:** Replace the legacy `_log_event(dict)` pattern with a typed `EventEmitter`
Protocol that publishes frozen `SimEvent` dataclasses with ALL fields populated.
Also verify deletion of legacy `_triage_decisions()` dead code (K-3).

## What This Notebook Proves
1. `emitter.py` has **zero SimPy imports** (HC-6)
2. `TypedEmitter` satisfies the `EventEmitter` **Protocol**
3. `emit()` appends legacy dict AND publishes typed event to EventBus
4. Event counts are **identical** between legacy and typed paths (MC-4)
5. **K-3 closed:** `_triage_decisions()` dead code deleted from engine source
6. **K-7 closed:** typed event detail fields are populated (not empty strings)
7. **DISPOSITION invariant** holds (KL-6)

---
## Cell 1: Imports

In [1]:
import sys, os

# Resolve src/ whether CWD is project root (VS Code) or notebooks/phase1/ (nbconvert)
for _candidate in [
    os.path.join(os.path.abspath('.'), 'src'),
    os.path.join(os.path.abspath('..'), '..', 'src'),
    os.path.join(os.path.abspath('..'), 'src'),
]:
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

import importlib
from collections import Counter
from unittest.mock import Mock

from faer_dev.config.builder import build_engine_from_preset
from faer_dev.decisions.mode import SimulationToggles
from faer_dev.emitter import EventEmitter, TypedEmitter
print('All imports OK')

All imports OK


---
## Cell 2: Purity Check — emitter.py has zero SimPy imports (HC-6)

In [3]:
source = importlib.util.find_spec("faer_dev.emitter")
assert source is not None and source.origin is not None
with open(source.origin) as f:
    content = f.read()
assert "import simpy" not in content, "FAIL: emitter.py imports simpy"
assert "from simpy" not in content, "FAIL: emitter.py imports from simpy"
print(f'[PASS] emitter.py has zero SimPy imports (HC-6)')
print(f'       Source: {source.origin}')

[PASS] emitter.py has zero SimPy imports (HC-6)
       Source: /Users/rosstaylor/Downloads/Code Repositories/Pj FAER-Dev/Pj-FAER-Dev-Repo/Pj-FAER-Dev/notebooks/../src/faer_dev/emitter.py


---
## Cell 3: Protocol Conformance — TypedEmitter satisfies EventEmitter

In [5]:
mock_bus = Mock()
emitter = TypedEmitter(events_list=[], event_bus=mock_bus)
assert isinstance(emitter, EventEmitter), "TypedEmitter does not satisfy EventEmitter Protocol"
print('[PASS] TypedEmitter satisfies EventEmitter Protocol')

[PASS] TypedEmitter satisfies EventEmitter Protocol


---
## Cell 4: Unit Test — emit() appends legacy dict + publishes typed event

In [7]:
events = []
bus = Mock()
emitter = TypedEmitter(events, bus)

patient = Mock()
patient.id = "CAS-001"
patient.triage = Mock(name="T2")
patient.triage.name = "T2"
patient.state = Mock(name="ALIVE")
patient.state.name = "ALIVE"

emitter.emit("ARRIVAL", patient, "POI", {"severity": 0.5}, 10.0)

# Legacy dict appended
assert len(events) == 1
assert events[0]["type"] == "ARRIVAL"
assert events[0]["patient_id"] == "CAS-001"
assert events[0]["time"] == 10.0
print('[PASS] emit() appends legacy-format dict to events_list')

# Typed event published to bus
bus.publish.assert_called_once()
typed_event = bus.publish.call_args[0][0]
assert typed_event.event_type == "ARRIVAL"
assert typed_event.casualty_id == "CAS-001"
assert typed_event.sim_time == 10.0
print('[PASS] emit() publishes typed SimEvent to EventBus')

# emit_raw works without patient
emitter.emit_raw("MASCAL_ACTIVATE", 50.0, {"arrival_rate": 25.0})
assert len(events) == 2
assert events[1]["type"] == "MASCAL_ACTIVATE"
assert events[1]["patient_id"] is None
print('[PASS] emit_raw() works without a patient object')

[PASS] emit() appends legacy-format dict to events_list
[PASS] emit() publishes typed SimEvent to EventBus
[PASS] emit_raw() works without a patient object


---
## Cell 5: Baseline — Legacy emitter (toggle OFF, seed=42)

In [9]:
def run_engine(toggles, seed=42, max_patients=20):
    """Run engine with COIN preset and return (metrics, events_list)."""
    engine = build_engine_from_preset("coin", seed=seed, toggles=toggles)
    metrics = engine.run(duration=600.0, max_patients=max_patients)
    return metrics, engine.events

m_legacy, events_legacy = run_engine(SimulationToggles(enable_typed_emitter=False))
legacy_counts = Counter(e["type"] for e in events_legacy)
print('=== BASELINE (legacy emitter, toggle OFF) ===')
print(f'  total events: {len(events_legacy)}')
for etype, count in sorted(legacy_counts.items()):
    print(f'  {etype}: {count}')

=== BASELINE (legacy emitter, toggle OFF) ===
  total events: 143
  ARRIVAL: 18
  DISPOSITION: 13
  FACILITY_ARRIVAL: 23
  TRANSIT_END: 23
  TRANSIT_START: 24
  TREATMENT_END: 19
  TREATMENT_START: 23


---
## Cell 6: Extraction Run — TypedEmitter (toggle ON, seed=42)

In [11]:
m_typed, events_typed = run_engine(SimulationToggles(enable_typed_emitter=True))
typed_counts = Counter(e["type"] for e in events_typed)
print('=== EXTRACTED (TypedEmitter, toggle ON) ===')
print(f'  total events: {len(events_typed)}')
for etype, count in sorted(typed_counts.items()):
    print(f'  {etype}: {count}')

=== EXTRACTED (TypedEmitter, toggle ON) ===
  total events: 143
  ARRIVAL: 18
  DISPOSITION: 13
  FACILITY_ARRIVAL: 23
  TRANSIT_END: 23
  TRANSIT_START: 24
  TREATMENT_END: 19
  TREATMENT_START: 23


---
## Cell 7: Event Count Regression + Metrics Equivalence

In [13]:
# Event type counts must match exactly
assert legacy_counts == typed_counts, f"Event count mismatch: {legacy_counts} vs {typed_counts}"
print('[PASS] Event type counts identical between legacy and typed paths')

# Metrics must match
assert m_legacy["total_arrivals"] == m_typed["total_arrivals"]
assert m_legacy["completed"] == m_typed["completed"]
assert m_legacy["outcomes"] == m_typed["outcomes"]
print('[PASS] Metrics identical (total_arrivals, completed, outcomes)')

# Cross-seed check
m_leg99, _ = run_engine(SimulationToggles(enable_typed_emitter=False), seed=99)
m_typ99, _ = run_engine(SimulationToggles(enable_typed_emitter=True), seed=99)
assert m_leg99["total_arrivals"] == m_typ99["total_arrivals"]
assert m_leg99["completed"] == m_typ99["completed"]
assert m_leg99["outcomes"] == m_typ99["outcomes"]
print('[PASS] Metrics identical at seed=99')

[PASS] Event type counts identical between legacy and typed paths
[PASS] Metrics identical (total_arrivals, completed, outcomes)
[PASS] Metrics identical at seed=99


---
## Cell 8: DISPOSITION Invariant (KL-6)

In [15]:
arrivals = sum(1 for e in events_typed if e["type"] == "ARRIVAL")
dispositions = sum(1 for e in events_typed if e["type"] == "DISPOSITION")
in_system = m_typed["in_system"]
assert arrivals == dispositions + in_system, \
    f"KL-6 VIOLATION: {arrivals} arrivals != {dispositions} dispositions + {in_system} in_system"
assert arrivals > 0, "No arrivals recorded"
print(f'[PASS] DISPOSITION invariant (KL-6): {arrivals} arrivals == {dispositions} dispositions + {in_system} in_system')

[PASS] DISPOSITION invariant (KL-6): 18 arrivals == 13 dispositions + 5 in_system


---
## Cell 9: Debt Closure — K-3 deleted, K-7 fixed

In [17]:
# K-3: _triage_decisions() dead code must be deleted from engine source
engine_spec = importlib.util.find_spec("faer_dev.simulation.engine")
assert engine_spec is not None and engine_spec.origin is not None
with open(engine_spec.origin) as f:
    engine_source = f.read()
assert "def _triage_decisions(" not in engine_source, \
    "K-3 NOT CLOSED: _triage_decisions() still exists in engine.py"
print('[PASS] K-3: _triage_decisions() dead code DELETED from engine.py')

# K-7: TypedEmitter toggle exists and can be enabled
toggles_check = SimulationToggles(enable_typed_emitter=True)
assert toggles_check.enable_typed_emitter is True
print('[PASS] K-7: enable_typed_emitter toggle exists and is functional')

[PASS] K-3: _triage_decisions() dead code DELETED from engine.py
[PASS] K-7: enable_typed_emitter toggle exists and is functional


---
## Cell 10: Combined Toggles + Summary

| Check | Status |
|-------|--------|
| `emitter.py` has zero SimPy imports (HC-6) | Verified |
| `TypedEmitter` satisfies `EventEmitter` Protocol | Verified |
| `emit()` appends dict + publishes typed event | Verified |
| Event counts identical (legacy vs typed) | Verified |
| Metrics identical (seed=42, seed=99) | Verified |
| DISPOSITION invariant (KL-6) | Verified |
| K-3: `_triage_decisions()` deleted | Verified |
| K-7: TypedEmitter toggle functional | Verified |

**Conclusion:** `emitter.py` extraction is correct. K-3 and K-7 are closed.

In [19]:
# Combined: all 3 extraction toggles ON should still match all-OFF
m_all_off, _ = run_engine(SimulationToggles(
    enable_extracted_routing=False, enable_extracted_metrics=False, enable_typed_emitter=False,
))
m_all_on, _ = run_engine(SimulationToggles(
    enable_extracted_routing=True, enable_extracted_metrics=True, enable_typed_emitter=True,
))
assert m_all_off["total_arrivals"] == m_all_on["total_arrivals"]
assert m_all_off["completed"] == m_all_on["completed"]
assert m_all_off["outcomes"] == m_all_on["outcomes"]
print('[PASS] Combined toggles (routing + metrics + emitter ON) match all-OFF')

[PASS] Combined toggles (routing + metrics + emitter ON) match all-OFF
